# Taller 02 — Aprendizaje Supervisado
## Clasificación: Reconocimiento de Dígitos Manuscritos

**Dataset:** `sklearn.datasets.load_digits()` (1,797 registros)  
**Target:** Dígito manuscrito (0–9) — Clasificación Multiclase  
**Autores:** Javier Daza Olivella · Pablo Jimeno Juca · María Sofía Uribe  


## 1. Importación de Librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Carga de Datos

`sklearn.datasets.load_digits()` contiene 1,797 imágenes 8×8 de dígitos manuscritos (0–9), representadas como vectores de 64 features de intensidad de píxel (valores enteros 0–16). Es un benchmark clásico de clasificación multiclase.

In [2]:
from pathlib import Path

possible_paths = [Path('../data/raw/digits.csv'), Path('data/raw/digits.csv')]
DATA_PATH = next((path for path in possible_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('No se encontro data/raw/digits.csv')

df = pd.read_csv(DATA_PATH)
feature_names = sorted([col for col in df.columns if col.startswith('pixel_')], key=lambda x: int(x.split('_')[1]))
X_raw = df[feature_names].values
y_raw = df['target'].values

print(f'Ruta usada: {DATA_PATH.resolve()}')
print(f'Shape: {df.shape}')
print(f'Clases: {sorted(df["target"].unique())}')
print(f'Valores de píxel — min: {X_raw.min()}, max: {X_raw.max()}')
print('\nDistribución de clases:')
print(df['target'].value_counts().sort_index())


Ruta usada: /Users/mariasofiauribe/Projects/02_Active/intro_ia_2026/Workshop_02/data/raw/digits.csv
Shape: (1797, 65)
Clases: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Valores de píxel — min: 0.0, max: 16.0

Distribución de clases:
target
0    178
1    182
2    177
3    183
4    181
5    182
6    181
7    179
8    174
9    180
Name: count, dtype: int64


## 3. Análisis Exploratorio (EDA)

### 3.1 Distribución de clases

In [3]:
class_counts = df['target'].value_counts().sort_index()
missing_values = int(df[feature_names].isna().sum().sum())
ratio = class_counts.max() / class_counts.min()

plt.figure(figsize=(8, 4))
plt.bar(class_counts.index, class_counts.values, color='teal', edgecolor='white')
plt.xlabel('Dígito')
plt.ylabel('Cantidad de muestras')
plt.title('Distribución de Clases — Digits')
plt.xticks(range(10))
plt.tight_layout()
plt.show()
print(f'Balance: min={class_counts.min()}, max={class_counts.max()}, ratio={ratio:.2f}')
print(f'Valores faltantes totales: {missing_values}')


Balance: min=174, max=183, ratio=1.05
Valores faltantes totales: 0


**Conclusión:** El dataset está muy bien balanceado — cada dígito tiene aproximadamente 180 muestras (ratio ~1.05), no se observan valores faltantes y no hay evidencia de sesgo fuerte por desbalance de clases.

### 3.2 Visualización de muestras

In [4]:
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for digit in range(10):
    samples = df[df['target'] == digit].head(2)
    for row, (_, sample) in enumerate(samples.iterrows()):
        img = sample[feature_names].values.reshape(8, 8)
        axes[row, digit].imshow(img, cmap='gray_r', interpolation='nearest')
        axes[row, digit].axis('off')
        if row == 0:
            axes[row, digit].set_title(str(digit), fontsize=12, fontweight='bold')
plt.suptitle('2 muestras por clase de dígito', y=1.02)
plt.tight_layout()
plt.show()

**Conclusión:** Los dígitos son claramente distinguibles a pesar de la baja resolución 8×8. Se observan variaciones de escritura entre muestras de la misma clase. Los dígitos 3, 5 y 8 comparten trazos curvos similares, lo que puede generar confusión entre clasificadores.

### 3.3 Correlación entre píxeles

In [5]:
pixel_variances = df[feature_names].var().sort_values(ascending=False)
top_pixels = pixel_variances.head(16).index.tolist()
corr_top = df[top_pixels].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_top, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlación entre los 16 píxeles con mayor varianza')
plt.tight_layout()
plt.show()
print('Píxeles analizados:', top_pixels)


Píxeles analizados:

 ['pixel_42', 'pixel_43', 'pixel_34', 'pixel_35', 'pixel_44', 'pixel_21', 'pixel_26', 'pixel_20', 'pixel_28', 'pixel_13', 'pixel_53', 'pixel_36', 'pixel_61', 'pixel_27', 'pixel_29', 'pixel_37']


**Conclusión:** Las correlaciones más altas aparecen entre píxeles vecinos de la zona central, donde se concentran los trazos del dígito. Esto confirma que existe estructura espacial útil para la clasificación.

### 3.4 Visualización PCA 2D


In [6]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_raw)
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_raw, cmap='tab10', alpha=0.6, s=20)
plt.colorbar(scatter, label='Dígito')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
plt.title(f'PCA 2D — Varianza explicada total: {sum(pca.explained_variance_ratio_):.1%}')
plt.tight_layout()
plt.show()

**Conclusión:** Aunque solo se explica ~28% de la varianza con 2 componentes, ya se observan agrupaciones distinguibles para varios dígitos. El solapamiento entre 3/5/8 y 4/9 confirma la dificultad del problema para métodos lineales simples.

### 3.5 Importancia de píxeles (preliminar)


In [7]:
rf_prelim = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_prelim.fit(X_raw, y_raw)
fi = rf_prelim.feature_importances_.reshape(8, 8)
plt.figure(figsize=(6, 5))
sns.heatmap(fi, cmap='YlOrRd', annot=False, cbar=True)
plt.title('Importancia de píxeles (mapa de calor 8×8)')
plt.xlabel('Columna píxel')
plt.ylabel('Fila píxel')
plt.tight_layout()
plt.show()

**Conclusión:** Los píxeles del centro de la imagen (filas 2-5, columnas 2-5) son los más discriminativos. Los bordes y esquinas tienen importancia mínima ya que la mayoría de los trazos pasan por el centro. Esta información podría usarse para una selección de features reducida.

## 4. Preprocesamiento

El dataset ya es numérico y no tiene valores faltantes. Solo se requiere:
- División estratificada train/test (80/20)
- `StandardScaler` dentro de cada Pipeline
- Mantener la misma matriz local `pixel_0 ... pixel_63` en notebooks y app


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Valores faltantes: {np.isnan(X_raw).sum()}")
print(f"\nDistribución en train: {np.bincount(y_train)}")
print(f"Distribución en test:  {np.bincount(y_test)}")

Train: (1437, 64), Test: (360, 64)
Valores faltantes: 0

Distribución en train: [142 146 142 146 145 145 145 143 139 144]
Distribución en test:  [36 36 35 37 36 37 36 36 35 36]


## 5. Entrenamiento y Ajuste de Hiperparámetros

Entrenamos 5 modelos con `StratifiedKFold(5)` y `GridSearchCV` optimizando accuracy. La estratificación asegura que cada fold mantenga la proporción de clases del dataset original.

In [9]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

# Logistic Regression
gs_lr = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, random_state=42))]),
    {'clf__C': [0.01, 0.1, 1, 10]}, cv=skf, scoring='accuracy', n_jobs=1)
gs_lr.fit(X_train, y_train)
best_lr = gs_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)
cv_lr = cross_val_score(best_lr, X_train, y_train, cv=skf, scoring='accuracy')
results['Logistic Regression'] = {
    'cv_acc': cv_lr.mean(), 'cv_std': cv_lr.std(),
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'f1_macro': f1_score(y_test, y_pred_lr, average='macro'),
    'f1_weighted': f1_score(y_test, y_pred_lr, average='weighted'),
    'cm': confusion_matrix(y_test, y_pred_lr),
    'best_params': gs_lr.best_params_,
}
print(f"LR done — best: {gs_lr.best_params_} | acc: {results['Logistic Regression']['accuracy']:.4f}")

# KNN
gs_knn = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier())]),
    {'clf__n_neighbors': [3, 5, 7], 'clf__weights': ['uniform', 'distance']},
    cv=skf, scoring='accuracy', n_jobs=1)
gs_knn.fit(X_train, y_train)
best_knn = gs_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test)
cv_knn = cross_val_score(best_knn, X_train, y_train, cv=skf, scoring='accuracy')
results['KNN'] = {
    'cv_acc': cv_knn.mean(), 'cv_std': cv_knn.std(),
    'accuracy': accuracy_score(y_test, y_pred_knn),
    'f1_macro': f1_score(y_test, y_pred_knn, average='macro'),
    'f1_weighted': f1_score(y_test, y_pred_knn, average='weighted'),
    'cm': confusion_matrix(y_test, y_pred_knn),
    'best_params': gs_knn.best_params_,
}
print(f"KNN done — best: {gs_knn.best_params_} | acc: {results['KNN']['accuracy']:.4f}")

# Decision Tree
gs_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'max_depth': [5, 10, 15, None], 'criterion': ['gini', 'entropy']},
    cv=skf, scoring='accuracy', n_jobs=1)
gs_dt.fit(X_train, y_train)
best_dt = gs_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test)
cv_dt = cross_val_score(best_dt, X_train, y_train, cv=skf, scoring='accuracy')
results['Decision Tree'] = {
    'cv_acc': cv_dt.mean(), 'cv_std': cv_dt.std(),
    'accuracy': accuracy_score(y_test, y_pred_dt),
    'f1_macro': f1_score(y_test, y_pred_dt, average='macro'),
    'f1_weighted': f1_score(y_test, y_pred_dt, average='weighted'),
    'cm': confusion_matrix(y_test, y_pred_dt),
    'best_params': gs_dt.best_params_,
    'fi': best_dt.feature_importances_,
}
print(f"DT done — best: {gs_dt.best_params_} | acc: {results['Decision Tree']['accuracy']:.4f}")

# Random Forest
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'n_estimators': [100, 200], 'max_depth': [None, 20]},
    cv=skf, scoring='accuracy', n_jobs=1)
gs_rf.fit(X_train, y_train)
best_rf = gs_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
cv_rf = cross_val_score(best_rf, X_train, y_train, cv=skf, scoring='accuracy')
results['Random Forest'] = {
    'cv_acc': cv_rf.mean(), 'cv_std': cv_rf.std(),
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'f1_macro': f1_score(y_test, y_pred_rf, average='macro'),
    'f1_weighted': f1_score(y_test, y_pred_rf, average='weighted'),
    'cm': confusion_matrix(y_test, y_pred_rf),
    'best_params': gs_rf.best_params_,
    'fi': best_rf.feature_importances_,
}
print(f"RF done — best: {gs_rf.best_params_} | acc: {results['Random Forest']['accuracy']:.4f}")

# Gradient Boosting
gs_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.1]},
    cv=skf, scoring='accuracy', n_jobs=1)
gs_gb.fit(X_train, y_train)
best_gb = gs_gb.best_estimator_
y_pred_gb = best_gb.predict(X_test)
cv_gb = cross_val_score(best_gb, X_train, y_train, cv=skf, scoring='accuracy')
results['Gradient Boosting'] = {
    'cv_acc': cv_gb.mean(), 'cv_std': cv_gb.std(),
    'accuracy': accuracy_score(y_test, y_pred_gb),
    'f1_macro': f1_score(y_test, y_pred_gb, average='macro'),
    'f1_weighted': f1_score(y_test, y_pred_gb, average='weighted'),
    'cm': confusion_matrix(y_test, y_pred_gb),
    'best_params': gs_gb.best_params_,
    'fi': best_gb.feature_importances_,
}
print(f"GB done — best: {gs_gb.best_params_} | acc: {results['Gradient Boosting']['accuracy']:.4f}")

LR done — best: {'clf__C': 1} | acc: 0.9722


KNN done — best: {'clf__n_neighbors': 3, 'clf__weights': 'uniform'} | acc: 0.9667


DT done — best: {'criterion': 'entropy', 'max_depth': 10} | acc: 0.8556


RF done — best: {'max_depth': None, 'n_estimators': 200} | acc: 0.9639


GB done — best: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100} | acc: 0.9556


## 6. Resultados y Evaluación

In [10]:
df_res = pd.DataFrame([
    {'Modelo': k, 'CV Acc (mean)': f"{v['cv_acc']:.4f}", 'CV Acc (std)': f"±{v['cv_std']:.4f}",
     'Test Accuracy': f"{v['accuracy']:.4f}", 'macro-F1': f"{v['f1_macro']:.4f}", 'weighted-F1': f"{v['f1_weighted']:.4f}"}
    for k, v in results.items()
])
print(df_res.to_string(index=False))

             Modelo CV Acc (mean) CV Acc (std) Test Accuracy macro-F1 weighted-F1
Logistic Regression        0.9680      ±0.0074        0.9722   0.9719      0.9722
                KNN        0.9763      ±0.0075        0.9667   0.9663      0.9666
      Decision Tree        0.8490      ±0.0210        0.8556   0.8542      0.8545
      Random Forest        0.9777      ±0.0087        0.9639   0.9634      0.9636
  Gradient Boosting        0.9589      ±0.0086        0.9556   0.9551      0.9553


In [11]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
cm = results[best_name]['cm']
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.title(f'Matriz de Confusión — {best_name}')
plt.tight_layout()
plt.show()
print(f"\nReporte de clasificación — {best_name}:")
preds_map = {'Logistic Regression': y_pred_lr, 'KNN': y_pred_knn,
             'Decision Tree': y_pred_dt, 'Random Forest': y_pred_rf, 'Gradient Boosting': y_pred_gb}
print(classification_report(y_test, preds_map[best_name]))


Reporte de clasificación — Logistic Regression:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       0.89      0.89      0.89        36
           2       1.00      1.00      1.00        35
           3       0.97      1.00      0.99        37
           4       0.97      1.00      0.99        36
           5       1.00      1.00      1.00        37
           6       1.00      0.97      0.99        36
           7       1.00      1.00      1.00        36
           8       0.89      0.89      0.89        35
           9       1.00      0.97      0.99        36

    accuracy                           0.97       360
   macro avg       0.97      0.97      0.97       360
weighted avg       0.97      0.97      0.97       360



In [12]:
fi_model = next((n for n in [best_name, 'Random Forest', 'Gradient Boosting', 'Decision Tree']
                 if 'fi' in results.get(n, {})), None)
if fi_model:
    fi_map = results[fi_model]['fi'].reshape(8, 8)
    plt.figure(figsize=(6, 5))
    sns.heatmap(fi_map, cmap='YlOrRd', annot=False, cbar=True)
    plt.title(f'Importancia de Píxeles — {fi_model}')
    plt.xlabel('Columna')
    plt.ylabel('Fila')
    plt.tight_layout()
    plt.show()

## 7. Conclusiones

1. **Todos los modelos logran alta accuracy** (>95% para LR, KNN y RF) demostrando que el dataset Digits es altamente separable.

2. **Logistic Regression** con regularización L2 y StandardScaler es competitiva con los ensambles, lo que indica que el problema es en gran parte linealmente separable en el espacio de píxeles normalizados.

3. **Decision Tree** tiene el peor desempeño, siendo propenso a overfitting en alta dimensionalidad (64 features) sin un ensamble.

4. **Confusiones frecuentes:** La matriz de confusión revela que los dígitos más confundidos son aquellos con trazos similares (3/5/8, 4/9, 7/1), coherente con la percepción humana.

5. **Importancia de píxeles:** Los píxeles centrales de la imagen 8×8 son los más informativos. Una reducción de dimensionalidad (PCA, selección de top-N píxeles) podría mantener alta accuracy con menor complejidad.

6. **Escalabilidad:** Para datasets de imágenes más grandes, las Redes Neuronales Convolucionales (CNN) superarían a estos modelos clásicos. Sin embargo, para 64 features los modelos de sklearn ofrecen excelente relación complejidad-rendimiento.

7. **Validación cruzada:** La baja desviación estándar de CV accuracy (~0.01) confirma que los resultados son estables y el modelo generaliza bien.